# FFT-accelerated beta-2 (L2) divergence experiments

> **Optional/legacy notebook.** The supported main experiments now use exact Gaussian formulas or direct spatial quadrature. This notebook retains the earlier periodic Fourier experiments and now contains only experiments based on the beta-2 divergence.

For the density-power-divergence convention, beta equal to 2 gives half the squared L2 distance:

$$
D_2(p,q)=\frac{1}{2}\int_{-\infty}^{\infty}(p(x)-q(x))^2\,\mathrm{d}x.
$$

For Gaussian KDEs based on the same sample, define the exact Gram matrix

$$
G_{ij}=\int \widehat f_{h_i}(x)\widehat f_{h_j}(x)\,\mathrm{d}x.
$$

Then

$$
D_2(\widehat f_{h_i},\widehat f_{h_j})
=\frac{1}{2}\left(G_{ii}+G_{jj}-2G_{ij}\right).
$$

The analytic Gaussian-KDE formula is exact but costs $O(N^2n^2)$. The experiments below instead evaluate the whole KDE family once by FFT and assemble $G$ with the matrix product $G \approx \Delta x\,FF^T$. The analytic helper is retained for the small validation check below. Control the FFT approximation with `beta2_half_width` and `beta2_spatial_points`.


In [52]:
from __future__ import annotations

import importlib

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from kde_tools.bandwidths import (
    gaussian_mise_oracle_bandwidth,
    nearest_bandwidth_grid_index,
)
from kde_tools.fourier import (
    beta2_divergence_matrix_fft,
    beta2_similarity_matrix_fft,
)
import kde_tools.similarities as kde_similarities

# Reload helpers that may have changed since this kernel was started.
importlib.reload(kde_similarities)
from kde_tools.similarities import (
    beta2_divergence_matrix,
    doob_stationary_mass,
    first_order_perron_vector,
    perron_vector,
    population_gaussian_beta2_divergence_matrix,
)

np.set_printoptions(precision=5, suppress=True)
seed = 20260818


## 1. Example uncorrected stationary distribution on a geometric grid

Choose the sample size, number of bandwidth candidates, bandwidth interval, and similarity scale below. The bandwidth candidates are uniformly spaced in $\log h$, so adjacent candidates differ by a constant multiplicative factor. The figure shows the uncorrected Doob stationary mass, with a purple dotted line marking its peak and a red dashed line marking the Gaussian MISE-oracle bandwidth.


In [70]:
stationary_example_seed = seed + 5
stationary_example_n = 200
stationary_example_N = 191
stationary_example_h_min = 0.01
stationary_example_h_max = 2.0
stationary_example_tau = 1.0
stationary_example_half_width = 12.0
stationary_example_spatial_points = 8192

if stationary_example_n < 1:
    raise ValueError("stationary_example_n must be positive.")
if stationary_example_N < 2:
    raise ValueError("stationary_example_N must be at least two.")
if not 0.0 < stationary_example_h_min < stationary_example_h_max:
    raise ValueError("Require 0 < h_min < h_max.")
if stationary_example_tau <= 0.0:
    raise ValueError("stationary_example_tau must be positive.")
if stationary_example_half_width <= 0.0:
    raise ValueError("stationary_example_half_width must be positive.")
if (
    stationary_example_spatial_points < 2
    or stationary_example_spatial_points % 2
):
    raise ValueError(
        "stationary_example_spatial_points must be a positive even integer."
    )

stationary_example_grid = np.geomspace(
    stationary_example_h_min,
    stationary_example_h_max,
    stationary_example_N,
)
stationary_example_rng = np.random.default_rng(stationary_example_seed)
stationary_example_samples = stationary_example_rng.normal(
    size=stationary_example_n
)
stationary_example_similarity = beta2_similarity_matrix_fft(
    samples=stationary_example_samples,
    h_grid=stationary_example_grid,
    tau=stationary_example_tau,
    half_width=stationary_example_half_width,
    spatial_points=stationary_example_spatial_points,
)
stationary_example_uncorrected_mass = doob_stationary_mass(
    stationary_example_similarity
)
stationary_example_peak_index = int(
    np.argmax(stationary_example_uncorrected_mass)
)
stationary_example_peak_h = float(
    stationary_example_grid[stationary_example_peak_index]
)
stationary_example_oracle_h = gaussian_mise_oracle_bandwidth(
    stationary_example_n
)
stationary_example_log_h_min = np.log10(stationary_example_h_min)
stationary_example_log_h_max = np.log10(stationary_example_h_max)
stationary_example_log_h_span = (
    stationary_example_log_h_max - stationary_example_log_h_min
)
stationary_example_peak_paper_x = (
    np.log10(stationary_example_peak_h)
    - stationary_example_log_h_min
) / stationary_example_log_h_span
stationary_example_oracle_paper_x = (
    np.log10(stationary_example_oracle_h)
    - stationary_example_log_h_min
) / stationary_example_log_h_span

stationary_example_uncorrected_figure = go.Figure(
    go.Scatter(
        x=stationary_example_grid,
        y=stationary_example_uncorrected_mass,
        mode="lines+markers",
        line=dict(color="#1f77b4", width=2.5),
        marker=dict(size=5),
        name="geometric grid",
        hovertemplate=(
            "geometric grid"
            "<br>h=%{x:.6g}"
            "<br>stationary mass=%{y:.6g}<extra></extra>"
        ),
    )
)
stationary_example_uncorrected_figure.add_vline(
    x=stationary_example_peak_h,
    line=dict(color="#9467bd", width=2, dash="dot"),
)
stationary_example_uncorrected_figure.add_vline(
    x=stationary_example_oracle_h,
    line=dict(color="#d62728", width=2, dash="dash"),
)
stationary_example_uncorrected_figure.add_annotation(
    x=stationary_example_peak_paper_x,
    y=1.0,
    xref="paper",
    yref="paper",
    text=f"grid peak: h={stationary_example_peak_h:.4g}",
    showarrow=False,
    xanchor="right",
    yanchor="bottom",
    font=dict(color="#9467bd"),
)
stationary_example_uncorrected_figure.add_annotation(
    x=stationary_example_oracle_paper_x,
    y=1.0,
    xref="paper",
    yref="paper",
    text=f"Gaussian MISE oracle: h={stationary_example_oracle_h:.4g}",
    showarrow=False,
    xanchor="left",
    yanchor="bottom",
    font=dict(color="#d62728"),
)
stationary_example_uncorrected_figure.update_xaxes(
    title_text="bandwidth h (geometric grid)",
    type="log",
    range=[
        stationary_example_log_h_min,
        stationary_example_log_h_max,
    ],
)
stationary_example_uncorrected_figure.update_yaxes(
    title_text="uncorrected Doob stationary mass",
    rangemode="tozero",
)
stationary_example_uncorrected_figure.update_layout(
    title=(
        "Uncorrected beta-2 stationary distribution on a geometric grid: "
        f"n={stationary_example_n}, N={stationary_example_N}"
    ),
    width=1050,
    height=550,
    template="plotly_white",
    showlegend=False,
    margin=dict(t=100, b=100),
)
stationary_example_uncorrected_figure.show()


### Linear-axis comparison

The following figure repeats the same Doob stationary distribution, selected grid peak, and Gaussian MISE oracle using a linear bandwidth axis.


In [71]:
stationary_example_linear_figure = go.Figure(
    stationary_example_uncorrected_figure
)
stationary_example_linear_figure.layout.annotations[0].update(
    x=stationary_example_peak_h,
    y=1.0,
    xref="x",
    yref="paper",
    xanchor="left",
)
stationary_example_linear_figure.layout.annotations[1].update(
    x=stationary_example_oracle_h,
    y=1.08,
    xref="x",
    yref="paper",
    xanchor="left",
)
stationary_example_linear_figure.update_xaxes(
    title_text="bandwidth h (linear axis)",
    type="linear",
    range=[stationary_example_h_min, stationary_example_h_max],
)
stationary_example_linear_figure.update_layout(
    title=(
        "Same beta-2 stationary distribution on a linear axis: "
        f"n={stationary_example_n}, N={stationary_example_N}"
    ),
    margin=dict(t=120, b=100),
)
stationary_example_linear_figure.show()


### Similarity scale versus chosen bandwidth

For the same sample and geometric bandwidth grid, the plot below varies the similarity scale $\lambda$ in $S_{ij}=\exp(-\lambda D_{2,ij})$ and records the bandwidth at the maximum of the uncorrected Doob stationary mass. Here $\lambda$ plays the same role as `stationary_example_tau` above. The red dashed horizontal line marks the Gaussian MISE-oracle bandwidth.


In [60]:
stationary_example_lambda_values = np.geomspace(1e-2, 1e4, 101)
stationary_example_log_lambda_min = np.log10(
    stationary_example_lambda_values[0]
)
stationary_example_log_lambda_max = np.log10(
    stationary_example_lambda_values[-1]
)
stationary_example_lambda_reference_paper_x = (
    np.log10(stationary_example_tau)
    - stationary_example_log_lambda_min
) / (
    stationary_example_log_lambda_max
    - stationary_example_log_lambda_min
)
stationary_example_oracle_paper_y = (
    np.log10(stationary_example_oracle_h)
    - stationary_example_log_h_min
) / stationary_example_log_h_span

stationary_example_divergence = beta2_divergence_matrix_fft(
    samples=stationary_example_samples,
    h_grid=stationary_example_grid,
    half_width=stationary_example_half_width,
    spatial_points=stationary_example_spatial_points,
)
stationary_example_lambda_selected_indices = np.empty(
    len(stationary_example_lambda_values),
    dtype=int,
)

for index, current_lambda in enumerate(
    stationary_example_lambda_values
):
    current_similarity = np.exp(
        -current_lambda * stationary_example_divergence
    )
    np.fill_diagonal(current_similarity, 1.0)
    current_mass = doob_stationary_mass(current_similarity)
    stationary_example_lambda_selected_indices[index] = int(
        np.argmax(current_mass)
    )

stationary_example_lambda_selected_bandwidths = (
    stationary_example_grid[
        stationary_example_lambda_selected_indices
    ]
)

stationary_example_lambda_figure = go.Figure(
    go.Scatter(
        x=stationary_example_lambda_values,
        y=stationary_example_lambda_selected_bandwidths,
        customdata=stationary_example_lambda_selected_indices,
        mode="lines+markers",
        line=dict(color="#1f77b4", width=2.5, shape="hv"),
        marker=dict(size=5),
        hovertemplate=(
            "lambda (λ)=%{x:.6g}"
            "<br>chosen bandwidth h=%{y:.6g}"
            "<br>grid index=%{customdata}<extra></extra>"
        ),
    )
)
stationary_example_lambda_figure.add_vline(
    x=stationary_example_tau,
    line=dict(color="#7f7f7f", width=2, dash="dot"),
)
stationary_example_lambda_figure.add_hline(
    y=stationary_example_oracle_h,
    line=dict(color="#d62728", width=2, dash="dash"),
)
stationary_example_lambda_figure.add_annotation(
    x=stationary_example_lambda_reference_paper_x,
    y=1.0,
    xref="paper",
    yref="paper",
    text=f"λ used above: {stationary_example_tau:g}",
    showarrow=False,
    xanchor="left",
    yanchor="bottom",
    font=dict(color="#7f7f7f"),
)
stationary_example_lambda_figure.add_annotation(
    x=1.0,
    y=stationary_example_oracle_paper_y,
    xref="paper",
    yref="paper",
    text=f"Gaussian MISE oracle: h={stationary_example_oracle_h:.4g}",
    showarrow=False,
    xanchor="right",
    yanchor="bottom",
    font=dict(color="#d62728"),
)
stationary_example_lambda_figure.update_xaxes(
    title_text="similarity scale λ",
    type="log",
    range=[
        stationary_example_log_lambda_min,
        stationary_example_log_lambda_max,
    ],
)
stationary_example_lambda_figure.update_yaxes(
    title_text="chosen bandwidth h",
    type="log",
    range=[
        stationary_example_log_h_min,
        stationary_example_log_h_max,
    ],
)
stationary_example_lambda_figure.update_layout(
    title=(
        "Similarity scale versus chosen beta-2 bandwidth: "
        f"n={stationary_example_n}, N={stationary_example_N}"
    ),
    width=1050,
    height=550,
    template="plotly_white",
    showlegend=False,
    margin=dict(t=100, b=100),
)
stationary_example_lambda_figure.show()


### Monte Carlo median selection and spread across samples

For each of the same similarity-scale values, this experiment draws $M=200$ independent samples of size $n$, records the chosen bandwidth-grid index, and plots its median. The shaded band is the interquartile range from the 25th to the 75th percentile. Each repetition reuses its sample across all $\lambda$ values, so changes along a repetition are paired.


In [61]:
stationary_example_mc_M = 200
stationary_example_mc_seed = stationary_example_seed + 1000

if stationary_example_mc_M < 1:
    raise ValueError("stationary_example_mc_M must be positive.")

stationary_example_mc_rng = np.random.default_rng(
    stationary_example_mc_seed
)
stationary_example_mc_selected_indices = np.empty(
    (
        stationary_example_mc_M,
        len(stationary_example_lambda_values),
    ),
    dtype=np.int16,
)

for repetition in range(stationary_example_mc_M):
    current_samples = stationary_example_mc_rng.normal(
        size=stationary_example_n
    )
    current_divergence = beta2_divergence_matrix_fft(
        samples=current_samples,
        h_grid=stationary_example_grid,
        half_width=stationary_example_half_width,
        spatial_points=stationary_example_spatial_points,
    )

    for lambda_index, current_lambda in enumerate(
        stationary_example_lambda_values
    ):
        current_similarity = np.exp(
            -current_lambda * current_divergence
        )
        np.fill_diagonal(current_similarity, 1.0)
        current_mass = doob_stationary_mass(current_similarity)
        stationary_example_mc_selected_indices[
            repetition, lambda_index
        ] = int(np.argmax(current_mass))

    if (repetition + 1) % 10 == 0:
        print(
            f"completed {repetition + 1}/"
            f"{stationary_example_mc_M} repetitions"
        )

(
    stationary_example_mc_q25_indices,
    stationary_example_mc_median_indices,
    stationary_example_mc_q75_indices,
) = np.quantile(
    stationary_example_mc_selected_indices,
    [0.25, 0.50, 0.75],
    axis=0,
)

stationary_example_index_grid = np.arange(
    stationary_example_N, dtype=float
)
stationary_example_log_bandwidth_grid = np.log(
    stationary_example_grid
)

def stationary_example_bandwidth_at_index(indices):
    return np.exp(
        np.interp(
            indices,
            stationary_example_index_grid,
            stationary_example_log_bandwidth_grid,
        )
    )

stationary_example_mc_q25_bandwidths = (
    stationary_example_bandwidth_at_index(
        stationary_example_mc_q25_indices
    )
)
stationary_example_mc_median_bandwidths = (
    stationary_example_bandwidth_at_index(
        stationary_example_mc_median_indices
    )
)
stationary_example_mc_q75_bandwidths = (
    stationary_example_bandwidth_at_index(
        stationary_example_mc_q75_indices
    )
)


completed 10/200 repetitions
completed 20/200 repetitions
completed 30/200 repetitions
completed 40/200 repetitions
completed 50/200 repetitions
completed 60/200 repetitions
completed 70/200 repetitions
completed 80/200 repetitions
completed 90/200 repetitions
completed 100/200 repetitions
completed 110/200 repetitions
completed 120/200 repetitions
completed 130/200 repetitions
completed 140/200 repetitions
completed 150/200 repetitions
completed 160/200 repetitions
completed 170/200 repetitions
completed 180/200 repetitions
completed 190/200 repetitions
completed 200/200 repetitions


In [62]:
stationary_example_mc_oracle_index = float(
    np.interp(
        np.log(stationary_example_oracle_h),
        stationary_example_log_bandwidth_grid,
        stationary_example_index_grid,
    )
)
stationary_example_mc_oracle_paper_y = (
    stationary_example_mc_oracle_index + 0.5
) / stationary_example_N
stationary_example_mc_tick_indices = np.unique(
    np.linspace(
        0, stationary_example_N - 1, 7, dtype=int
    )
)
stationary_example_mc_tick_labels = [
    f"{index}<br>h={stationary_example_grid[index]:.3g}"
    for index in stationary_example_mc_tick_indices
]

stationary_example_mc_figure = go.Figure()
stationary_example_mc_figure.add_trace(
    go.Scatter(
        x=stationary_example_lambda_values,
        y=stationary_example_mc_q25_indices,
        customdata=stationary_example_mc_q25_bandwidths,
        mode="lines",
        line=dict(color="rgba(31,119,180,0)", shape="hv"),
        name="25th percentile",
        showlegend=False,
        hovertemplate=(
            "λ=%{x:.6g}"
            "<br>25th-percentile index=%{y:.3g}"
            "<br>corresponding h=%{customdata:.6g}<extra></extra>"
        ),
    )
)
stationary_example_mc_figure.add_trace(
    go.Scatter(
        x=stationary_example_lambda_values,
        y=stationary_example_mc_q75_indices,
        customdata=stationary_example_mc_q75_bandwidths,
        mode="lines",
        line=dict(color="rgba(31,119,180,0)", shape="hv"),
        fill="tonexty",
        fillcolor="rgba(31,119,180,0.22)",
        name="interquartile range",
        hovertemplate=(
            "λ=%{x:.6g}"
            "<br>75th-percentile index=%{y:.3g}"
            "<br>corresponding h=%{customdata:.6g}<extra></extra>"
        ),
    )
)
stationary_example_mc_figure.add_trace(
    go.Scatter(
        x=stationary_example_lambda_values,
        y=stationary_example_mc_median_indices,
        customdata=stationary_example_mc_median_bandwidths,
        mode="lines+markers",
        line=dict(color="#1f77b4", width=2.5, shape="hv"),
        marker=dict(size=5),
        name="median selected index",
        hovertemplate=(
            "λ=%{x:.6g}"
            "<br>median index=%{y:.3g}"
            "<br>corresponding h=%{customdata:.6g}<extra></extra>"
        ),
    )
)
stationary_example_mc_figure.add_vline(
    x=stationary_example_tau,
    line=dict(color="#7f7f7f", width=2, dash="dot"),
)
stationary_example_mc_figure.add_hline(
    y=stationary_example_mc_oracle_index,
    line=dict(color="#d62728", width=2, dash="dash"),
)
stationary_example_mc_figure.add_annotation(
    x=stationary_example_lambda_reference_paper_x,
    y=1.0,
    xref="paper",
    yref="paper",
    text=f"λ used above: {stationary_example_tau:g}",
    showarrow=False,
    xanchor="left",
    yanchor="bottom",
    font=dict(color="#7f7f7f"),
)
stationary_example_mc_figure.add_annotation(
    x=1.0,
    y=stationary_example_mc_oracle_paper_y,
    xref="paper",
    yref="paper",
    text=(
        "Gaussian MISE oracle: "
        f"index={stationary_example_mc_oracle_index:.2f}, "
        f"h={stationary_example_oracle_h:.4g}"
    ),
    showarrow=False,
    xanchor="right",
    yanchor="bottom",
    font=dict(color="#d62728"),
)
stationary_example_mc_figure.update_xaxes(
    title_text="similarity scale λ",
    type="log",
    range=[
        stationary_example_log_lambda_min,
        stationary_example_log_lambda_max,
    ],
)
stationary_example_mc_figure.update_yaxes(
    title_text="chosen bandwidth grid index",
    range=[-0.5, stationary_example_N - 0.5],
    tickmode="array",
    tickvals=stationary_example_mc_tick_indices,
    ticktext=stationary_example_mc_tick_labels,
)
stationary_example_mc_figure.update_layout(
    title=(
        "Monte Carlo median beta-2 bandwidth index versus λ: "
        f"n={stationary_example_n}, N={stationary_example_N}, "
        f"M={stationary_example_mc_M}"
    ),
    width=1050,
    height=600,
    template="plotly_white",
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.18,
        yanchor="top",
    ),
    margin=dict(t=100, b=150),
)
stationary_example_mc_figure.show()


## 2. Configurable uncorrected beta-2 selection histogram

Choose $n$, $N$, $M$, the bandwidth interval, and $\tau$ below. Each repetition selects the largest component of the uncorrected Doob stationary distribution on the fixed geometric bandwidth grid.


In [63]:
beta2_n = 200
beta2_N = 201
beta2_M = 100

beta2_h_min = 0.01
beta2_h_max = 2.0
beta2_tau = 1.0
beta2_seed = seed + 10
beta2_half_width = 12.0
beta2_spatial_points = 8192
beta2_validate_fft = True

if beta2_n < 1 or beta2_M < 1:
    raise ValueError("beta2_n and beta2_M must be positive.")
if beta2_N < 2:
    raise ValueError("beta2_N must be at least 2.")
if not 0.0 < beta2_h_min < beta2_h_max:
    raise ValueError("Require 0 < beta2_h_min < beta2_h_max.")
if beta2_tau <= 0.0:
    raise ValueError("beta2_tau must be positive.")
if beta2_half_width <= 0.0:
    raise ValueError("beta2_half_width must be positive.")
if beta2_spatial_points < 2:
    raise ValueError("beta2_spatial_points must be at least 2.")

beta2_h_grid = np.geomspace(
    beta2_h_min,
    beta2_h_max,
    beta2_N,
)

print(
    f"beta=2, n={beta2_n}, N={beta2_N}, M={beta2_M}, "
    f"tau={beta2_tau:g}"
)

beta=2, n=200, N=201, M=100, tau=1


In [64]:
if beta2_validate_fft:
    validation_rng = np.random.default_rng(beta2_seed)
    validation_samples = validation_rng.normal(
        size=min(beta2_n, 100)
    )
    validation_indices = np.unique(
        np.linspace(0, beta2_N - 1, min(beta2_N, 20), dtype=int)
    )
    validation_h_grid = beta2_h_grid[validation_indices]
    validation_exact = beta2_divergence_matrix(
        samples=validation_samples,
        h_grid=validation_h_grid,
    )
    validation_fft = beta2_divergence_matrix_fft(
        samples=validation_samples,
        h_grid=validation_h_grid,
        half_width=beta2_half_width,
        spatial_points=beta2_spatial_points,
    )
    beta2_fft_maximum_absolute_error = float(
        np.max(np.abs(validation_fft - validation_exact))
    )
    print(
        "FFT validation maximum absolute error: "
        f"{beta2_fft_maximum_absolute_error:.3e}"
    )

beta2_rng = np.random.default_rng(beta2_seed)
beta2_selected_indices = np.empty(beta2_M, dtype=int)

for repetition in range(beta2_M):
    current_samples = beta2_rng.normal(size=beta2_n)
    current_similarity = beta2_similarity_matrix_fft(
        samples=current_samples,
        h_grid=beta2_h_grid,
        tau=beta2_tau,
        half_width=beta2_half_width,
        spatial_points=beta2_spatial_points,
    )
    current_stationary = doob_stationary_mass(current_similarity)
    beta2_selected_indices[repetition] = int(
        np.argmax(current_stationary)
    )

    if (repetition + 1) % 5 == 0 or repetition + 1 == beta2_M:
        print(f"completed {repetition + 1}/{beta2_M} repetitions")

beta2_selection_counts = np.bincount(
    beta2_selected_indices,
    minlength=beta2_N,
)


FFT validation maximum absolute error: 2.176e-14
completed 5/100 repetitions
completed 10/100 repetitions
completed 15/100 repetitions
completed 20/100 repetitions
completed 25/100 repetitions
completed 30/100 repetitions
completed 35/100 repetitions
completed 40/100 repetitions
completed 45/100 repetitions
completed 50/100 repetitions
completed 55/100 repetitions
completed 60/100 repetitions
completed 65/100 repetitions
completed 70/100 repetitions
completed 75/100 repetitions
completed 80/100 repetitions
completed 85/100 repetitions
completed 90/100 repetitions
completed 95/100 repetitions
completed 100/100 repetitions


In [65]:
beta2_tick_indices = np.unique(
    np.linspace(0, beta2_N - 1, 7, dtype=int)
)
beta2_tick_bandwidths = [
    f"{beta2_h_grid[index]:.3g}" for index in beta2_tick_indices
]
beta2_nonzero_indices = np.flatnonzero(beta2_selection_counts)

beta2_histogram = go.Figure(
    go.Bar(
        x=beta2_nonzero_indices,
        y=beta2_selection_counts[beta2_nonzero_indices],
        customdata=beta2_h_grid[beta2_nonzero_indices],
        width=0.85,
        marker=dict(
            color="#1f77b4",
            line=dict(color="white", width=0.5),
        ),
        hovertemplate=(
            "grid index=%{x}"
            "<br>bandwidth h=%{customdata:.6g}"
            "<br>frequency=%{y}<extra></extra>"
        ),
    )
)
beta2_histogram.update_xaxes(
    title_text="selected bandwidth h (geometric grid)",
    range=[-0.5, beta2_N - 0.5],
    tickmode="array",
    tickvals=beta2_tick_indices,
    ticktext=beta2_tick_bandwidths,
)
beta2_histogram.update_yaxes(
    title_text=f"frequency among {beta2_M} repetitions",
    rangemode="tozero",
    dtick=1,
)
beta2_histogram.update_layout(
    title=(
        f"Uncorrected Doob beta-2 selections: n={beta2_n}, "
        f"N={beta2_N}, M={beta2_M}, tau={beta2_tau:g}"
    ),
    width=950,
    height=500,
    template="plotly_white",
    showlegend=False,
)
beta2_histogram.show()


In [63]:
beta2_tau = 10.0

## 3. Uncorrected beta-2 histograms for increasing sample size

This repeats the FFT-accelerated uncorrected selection for $n=100,200,\ldots,1000$ on one fixed geometric grid. Prefixes of the same maximum-size sample are reused within each repetition. Dashed vertical lines mark the nearest grid indices to the geometric center and the Gaussian MISE-oracle bandwidth.


In [66]:
beta2_n_sweep_values = np.arange(100, 1001, 100, dtype=int)
beta2_n_sweep_N = 201
beta2_n_sweep_M = 500
beta2_n_sweep_h_min = beta2_h_min
beta2_n_sweep_h_max = beta2_h_max
beta2_n_sweep_tau = beta2_tau
beta2_n_sweep_seed = beta2_seed + 2

beta2_n_sweep_h_grid = np.geomspace(
    beta2_n_sweep_h_min,
    beta2_n_sweep_h_max,
    beta2_n_sweep_N,
)
beta2_n_sweep_center_h = np.sqrt(
    beta2_n_sweep_h_min * beta2_n_sweep_h_max
)
beta2_n_sweep_center_index = nearest_bandwidth_grid_index(
    beta2_n_sweep_center_h,
    beta2_n_sweep_h_grid,
)
beta2_n_sweep_oracle_bandwidths = {
    int(current_n): gaussian_mise_oracle_bandwidth(int(current_n))
    for current_n in beta2_n_sweep_values
}
beta2_n_sweep_oracle_indices = {
    int(current_n): nearest_bandwidth_grid_index(
        beta2_n_sweep_oracle_bandwidths[int(current_n)],
        beta2_n_sweep_h_grid,
    )
    for current_n in beta2_n_sweep_values
}

beta2_n_sweep_rng = np.random.default_rng(beta2_n_sweep_seed)
beta2_n_sweep_full_samples = beta2_n_sweep_rng.normal(
    size=(beta2_n_sweep_M, int(np.max(beta2_n_sweep_values)))
)
beta2_n_sweep_results = {}

for current_n in beta2_n_sweep_values:
    selected_indices = np.empty(beta2_n_sweep_M, dtype=int)

    for repetition in range(beta2_n_sweep_M):
        current_samples = beta2_n_sweep_full_samples[
            repetition, : int(current_n)
        ]
        current_similarity = beta2_similarity_matrix_fft(
            samples=current_samples,
            h_grid=beta2_n_sweep_h_grid,
            tau=beta2_n_sweep_tau,
            half_width=beta2_half_width,
            spatial_points=beta2_spatial_points,
        )
        selected_indices[repetition] = int(
            np.argmax(doob_stationary_mass(current_similarity))
        )

    selection_counts = np.bincount(
        selected_indices,
        minlength=beta2_n_sweep_N,
    )
    beta2_n_sweep_results[int(current_n)] = {
        "selected_indices": selected_indices.copy(),
        "counts": selection_counts,
        "relative_frequencies": selection_counts / beta2_n_sweep_M,
        "oracle_bandwidth": beta2_n_sweep_oracle_bandwidths[
            int(current_n)
        ],
        "oracle_index": beta2_n_sweep_oracle_indices[int(current_n)],
    }
    print(f"completed n={current_n}")


completed n=100
completed n=200
completed n=300
completed n=400
completed n=500
completed n=600
completed n=700
completed n=800
completed n=900
completed n=1000


In [67]:
beta2_n_rows = len(beta2_n_sweep_values)
beta2_n_ticks = np.unique(
    np.linspace(0, beta2_n_sweep_N - 1, 7, dtype=int)
)
beta2_n_tick_labels = [
    f"{beta2_n_sweep_h_grid[index]:.3g}" for index in beta2_n_ticks
]
beta2_n_max_frequency = max(
    float(result["relative_frequencies"].max())
    for result in beta2_n_sweep_results.values()
)
beta2_n_line_height = 1.05 * beta2_n_max_frequency

beta2_n_stacked = make_subplots(
    rows=beta2_n_rows,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[f"n={value}" for value in beta2_n_sweep_values],
    vertical_spacing=0.012,
)
beta2_n_overlay = go.Figure()

for row, current_n in enumerate(beta2_n_sweep_values, start=1):
    result = beta2_n_sweep_results[int(current_n)]
    nonzero_indices = np.flatnonzero(result["counts"])
    relative_frequencies = result["relative_frequencies"][nonzero_indices]

    beta2_n_stacked.add_trace(
        go.Bar(
            x=nonzero_indices,
            y=relative_frequencies,
            customdata=beta2_n_sweep_h_grid[nonzero_indices],
            width=0.85,
            marker=dict(
                color="#1f77b4",
                line=dict(color="white", width=0.35),
            ),
            opacity=0.8,
            name="uncorrected Doob selections",
            legendgroup="beta2-uncorrected",
            hovertemplate=(
                f"uncorrected, n={current_n}"
                "<br>h=%{customdata:.6g}"
                "<br>relative frequency=%{y:.1%}<extra></extra>"
            ),
            showlegend=(row == 1),
        ),
        row=row,
        col=1,
    )

    for x_index, name, color, legend_group in (
        (
            beta2_n_sweep_center_index,
            "geometric center (nearest index)",
            "#555555",
            "beta2-center",
        ),
        (
            result["oracle_index"],
            "Gaussian MISE oracle (nearest index)",
            "#d62728",
            "beta2-oracle",
        ),
    ):
        beta2_n_stacked.add_trace(
            go.Scatter(
                x=[x_index, x_index],
                y=[0.0, beta2_n_line_height],
                mode="lines",
                line=dict(color=color, width=2, dash="dash"),
                name=name,
                legendgroup=legend_group,
                showlegend=(row == 1),
            ),
            row=row,
            col=1,
        )
    beta2_n_stacked.update_yaxes(
        range=[0.0, beta2_n_line_height],
        tickformat=".0%",
        nticks=4,
        row=row,
        col=1,
    )

    color = f"hsl({360.0 * (row - 1) / beta2_n_rows:.1f},65%,45%)"
    beta2_n_overlay.add_trace(
        go.Bar(
            x=nonzero_indices,
            y=relative_frequencies,
            customdata=beta2_n_sweep_h_grid[nonzero_indices],
            width=0.85,
            marker_color=color,
            opacity=0.22,
            name=f"n={current_n}",
            hovertemplate=(
                f"uncorrected, n={current_n}"
                "<br>h=%{customdata:.6g}"
                "<br>relative frequency=%{y:.1%}<extra></extra>"
            ),
        )
    )

beta2_n_stacked.update_xaxes(
    range=[-0.5, beta2_n_sweep_N - 0.5],
    tickmode="array",
    tickvals=beta2_n_ticks,
    ticktext=beta2_n_tick_labels,
)
beta2_n_stacked.update_xaxes(
    title_text="selected bandwidth h (geometric grid)",
    row=beta2_n_rows,
    col=1,
)
beta2_n_stacked.update_layout(
    title=(
        f"Uncorrected FFT beta-2 selections for increasing n: "
        f"N={beta2_n_sweep_N}, M={beta2_n_sweep_M}"
    ),
    width=1050,
    height=230 * beta2_n_rows + 170,
    template="plotly_white",
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.025),
    margin=dict(t=100, b=160),
)
beta2_n_stacked.show()

beta2_n_overlay.update_xaxes(
    title_text="selected bandwidth h (geometric grid)",
    range=[-0.5, beta2_n_sweep_N - 0.5],
    tickmode="array",
    tickvals=beta2_n_ticks,
    ticktext=beta2_n_tick_labels,
)
beta2_n_overlay.update_yaxes(
    title_text="relative frequency",
    tickformat=".0%",
)
beta2_n_overlay.update_layout(
    title=(
        f"Overlay of uncorrected FFT beta-2 histograms: "
        f"N={beta2_n_sweep_N}, M={beta2_n_sweep_M}"
    ),
    width=1100,
    height=650,
    template="plotly_white",
    barmode="overlay",
    legend=dict(title="sample size"),
)
beta2_n_overlay.show()


## 4. Sample, exact-population, and first-order Perron vectors

This experiment compares three sum-normalized Perron vectors on the same bandwidth grid:

1. the sample vector obtained from the FFT beta-2 similarity matrix;
2. the exact population vector obtained by assembling the analytic Gaussian population matrix and solving its Perron eigenproblem numerically; and
3. the first-order perturbation formula

$$
v_i^{(1)}
=\frac{1}{N}
-\frac{\tau}{N^2}\left(r_i-\bar r\right),
\qquad
r_i=\sum_j D_{ij}^{\mathrm{pop}}.
$$

The implicit row-sum characterization determines the maximum of the first-order vector; it does not by itself provide the full exact Perron vector. All curves are displayed as $Nv_i$, so the horizontal value one corresponds to the uniform vector.

In [68]:
perron_comparison_n = 1000
perron_comparison_N = 191
perron_comparison_h_min = 0.01
perron_comparison_h_max = 2.0
perron_comparison_tau = 50.0
# Match the first repetition of the beta-2 sample-size sweep.
perron_comparison_seed = seed + 12
perron_comparison_half_width = beta2_half_width
perron_comparison_spatial_points = beta2_spatial_points

if perron_comparison_n < 1 or perron_comparison_N < 2:
    raise ValueError("Require n >= 1 and N >= 2.")
if not 0.0 < perron_comparison_h_min < perron_comparison_h_max:
    raise ValueError("Require 0 < h_min < h_max.")
if perron_comparison_tau < 0.0:
    raise ValueError("perron_comparison_tau must be nonnegative.")

perron_comparison_h_grid = np.geomspace(
    perron_comparison_h_min,
    perron_comparison_h_max,
    perron_comparison_N,
)
perron_comparison_rng = np.random.default_rng(perron_comparison_seed)
perron_comparison_samples = perron_comparison_rng.normal(
    size=perron_comparison_n
)

perron_comparison_sample_similarity = beta2_similarity_matrix_fft(
    samples=perron_comparison_samples,
    h_grid=perron_comparison_h_grid,
    tau=perron_comparison_tau,
    half_width=perron_comparison_half_width,
    spatial_points=perron_comparison_spatial_points,
)
perron_comparison_sample_vector = perron_vector(
    perron_comparison_sample_similarity
)

perron_comparison_population_divergence = (
    population_gaussian_beta2_divergence_matrix(
        h_grid=perron_comparison_h_grid,
        target_sigma=1.0,
    )
)
perron_comparison_population_similarity = np.exp(
    -perron_comparison_tau
    * perron_comparison_population_divergence
)
np.fill_diagonal(perron_comparison_population_similarity, 1.0)
perron_comparison_population_vector = perron_vector(
    perron_comparison_population_similarity
)

perron_comparison_first_order_vector = first_order_perron_vector(
    divergence_matrix=perron_comparison_population_divergence,
    tau=perron_comparison_tau,
)
perron_comparison_row_sums = (
    perron_comparison_population_divergence.sum(axis=1)
)
perron_comparison_implicit_index = int(
    np.argmin(perron_comparison_row_sums)
)

perron_comparison_vectors = {
    "sample FFT": perron_comparison_sample_vector,
    "exact population": perron_comparison_population_vector,
    "first-order population": perron_comparison_first_order_vector,
}
perron_comparison_selected_indices = {
    name: int(np.argmax(vector))
    for name, vector in perron_comparison_vectors.items()
}

for name, selected_index in perron_comparison_selected_indices.items():
    print(
        f"{name}: index={selected_index}, "
        f"h={perron_comparison_h_grid[selected_index]:.6g}"
    )
print(
    "implicit row-sum characterization: "
    f"index={perron_comparison_implicit_index}, "
    f"h={perron_comparison_h_grid[perron_comparison_implicit_index]:.6g}"
)
print(
    "maximum population first-order error: "
    f"{np.max(np.abs(perron_comparison_population_vector - perron_comparison_first_order_vector)):.3e}"
)

sample FFT: index=80, h=0.0930796
exact population: index=124, h=0.317486
first-order population: index=138, h=0.46911
implicit row-sum characterization: index=138, h=0.46911
maximum population first-order error: 3.058e-03


In [69]:
perron_comparison_colors = {
    "sample FFT": "#1f77b4",
    "exact population": "#d62728",
    "first-order population": "#2ca02c",
}
perron_comparison_dashes = {
    "sample FFT": "solid",
    "exact population": "solid",
    "first-order population": "dash",
}

perron_comparison_figure = go.Figure()
for name, vector in perron_comparison_vectors.items():
    selected_index = perron_comparison_selected_indices[name]
    relative_vector = perron_comparison_N * vector
    perron_comparison_figure.add_trace(
        go.Scatter(
            x=perron_comparison_h_grid,
            y=relative_vector,
            customdata=vector,
            mode="lines",
            line=dict(
                color=perron_comparison_colors[name],
                width=2.5,
                dash=perron_comparison_dashes[name],
            ),
            name=name,
            hovertemplate=(
                f"{name}"
                "<br>h=%{x:.6g}"
                "<br>N v(h)=%{y:.6g}"
                "<br>v(h)=%{customdata:.8f}<extra></extra>"
            ),
        )
    )
    perron_comparison_figure.add_trace(
        go.Scatter(
            x=[perron_comparison_h_grid[selected_index]],
            y=[relative_vector[selected_index]],
            mode="markers",
            marker=dict(
                color=perron_comparison_colors[name],
                size=10,
                symbol="diamond",
                line=dict(color="white", width=1),
            ),
            name=f"{name} maximum",
            showlegend=False,
            hovertemplate=(
                f"{name} maximum"
                f"<br>index={selected_index}"
                "<br>h=%{x:.6g}"
                "<br>N v(h)=%{y:.6g}<extra></extra>"
            ),
        )
    )

perron_comparison_figure.add_hline(
    y=1.0,
    line=dict(color="#777777", width=1.5, dash="dot"),
    annotation_text="uniform vector",
    annotation_position="bottom right",
)
perron_comparison_figure.update_xaxes(
    title_text="bandwidth h (logarithmic scale)",
    type="log",
    range=[
        np.log10(perron_comparison_h_min),
        np.log10(perron_comparison_h_max),
    ],
)
perron_comparison_figure.update_yaxes(
    title_text="relative Perron component N v(h) (uniform = 1)",
)
perron_comparison_figure.update_layout(
    title=(
        f"Sample, exact-population, and first-order beta-2 Perron vectors: "
        f"n={perron_comparison_n}, N={perron_comparison_N}, "
        f"tau={perron_comparison_tau:g}"
    ),
    width=1100,
    height=650,
    template="plotly_white",
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.18,
    ),
    margin=dict(b=130),
)
perron_comparison_figure.show()